# Customer Churn Prediction
## Notebook 1 of 8 — Data Exploration

This project predicts which telecom customers are likely to **churn** (cancel their service) so the business can reach them with retention offers *before* they leave. The eight notebooks run in order: exploration, cleaning, EDA, feature engineering, a baseline model, evaluation, improvement, and the final model.

**Business problem:** Keeping an existing customer costs less than winning a new one. This telecom loses about **27% of its customers**, so the question is which of them to spend a retention offer on.

This is a portfolio project built on a public dataset. There is no real client, and the 70% recall target used later is one I set myself.

**Tools:** Python · pandas · Matplotlib · Seaborn · scikit-learn · XGBoost · SMOTE (imbalanced-learn) · joblib

> **This notebook:** first contact with the raw data — we check its shape, data types, hidden missing values, and how imbalanced the target is.

**Author:** La Yaung Linn Lett  &nbsp;·&nbsp;  **Last updated:** August 2026

---

## 1. Load the raw data

**What:** Read the original Telco dataset into a pandas DataFrame.

**Why:** Before cleaning or modelling anything, we need to see what we are actually working with — the columns, the values, and the overall shape of the problem.

**Expectation:** A wide table mixing customer demographics, the services they subscribe to, billing information, and a `Churn` target column.

In [1]:
# Standard library
import warnings

# Third-party
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

# Load the original, never-modified dataset from data/raw
df = pd.read_csv("../data/raw/telco-customer-churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


The dataset holds one row per customer: demographics (`gender`, `SeniorCitizen`, `Partner`, `Dependents`), account details (`tenure`, `Contract`, `PaymentMethod`), the services they use, and their billing (`MonthlyCharges`, `TotalCharges`). The `Churn` column (Yes/No) is what we ultimately want to predict.

## 2. How big is the dataset?

**What:** Check the number of rows and columns.

**Why:** Sample size tells us how much we can trust the model and whether we can afford a held-out test set.

In [2]:
# (rows, columns)
df.shape

(7043, 21)

**7,043 customers × 21 columns.** That is comfortably enough data to train a model and still hold back ~1,400 rows for honest testing.

## 3. Data types and a hidden data-quality trap

**What:** Inspect column data types and look for missing values.

**Why:** Wrong data types and missing values silently break models, so we catch them now.

**What to watch:** `TotalCharges` *looks* numeric but may have been read as text.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

Notice that **`TotalCharges` is stored as text (`object`/`str`), not a number.** Every other charge column is numeric. That is a red flag — something non-numeric is hiding in that column.

In [4]:
# pandas reports zero nulls...
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

pandas reports **zero missing values** — but that is misleading. The missing values in `TotalCharges` are stored as blank spaces (`' '`), which pandas does not count as null. Let's confirm that.

In [5]:
# The 'missing' TotalCharges values are actually blank strings
blank_total_charges = df[df["TotalCharges"] == " "]
print(f"Rows with blank TotalCharges: {len(blank_total_charges)}")
blank_total_charges[["customerID", "tenure", "TotalCharges", "Churn"]]

Rows with blank TotalCharges: 11


,customerID,tenure,TotalCharges,Churn
488,4472-LVYGI,0,,No
753,3115-CZMZD,0,,No
936,5709-LVOEQ,0,,No
1082,4367-NUYAO,0,,No
1340,1371-DWPAZ,0,,No
3331,7644-OMVMY,0,,No
3826,3213-VVOLG,0,,No
4380,2520-SGTTA,0,,No
5218,2923-ARZLG,0,,No
6670,4075-WKNIU,0,,No


**11 rows** have a blank `TotalCharges` — and every one of them has `tenure = 0`. These are brand-new customers who have not been billed yet, so a total charge of **0** is the logical fill value. We will fix this in the cleaning notebook.

## 4. How imbalanced is the target?

**What:** Count how many customers churned vs. stayed.

**Why:** Class imbalance is the single most important fact about this problem. If churners are rare, plain accuracy becomes a misleading metric and we will need to design around it.

In [6]:
churn_counts = df["Churn"].value_counts()
churn_rate = df["Churn"].value_counts(normalize=True).round(3)
print(churn_counts)
print()
print(churn_rate)

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


**About 73% of customers stayed and only ~27% churned.** This is a clear class imbalance. A lazy model that predicts "no one churns" would still be ~73% accurate while being completely useless — which is exactly why later notebooks focus on **recall** (catching real churners), not raw accuracy.

## Section conclusion — what exploration told us

- The dataset is **7,043 customers × 21 columns**, mixing demographics, services, and billing.
- **`TotalCharges` is mis-typed as text** and hides **11 blank values**, all belonging to brand-new (`tenure = 0`) customers.
- pandas reports no nulls, proving that real-world "missing" data can be disguised as blank strings.
- The target is **imbalanced (~73% No / ~27% Yes)**, so accuracy alone will not be a trustworthy metric.

**Next:** Notebook 02 cleans these issues and saves a tidy dataset for the rest of the pipeline.